# Prepare the data

In [1]:
ls -lh /app/data/custom_data_1/

total 13G
-rwxrwxrwx 1 1018 1018 517M Jul 17 13:17 ERA5-20250716T095623Z-1-001.zip*
drwxrwxrwx 2 root root 4.0K Jul 22 11:40 ERA5_preprocessed/
drwxrwxrwx 3 root root 4.0K Aug  5 08:50 ERA5_wrf_combined/
-rwxrwxrwx 1 1018 1018 2.0G Jul 17 13:13 WRF-20250716T095739Z-1-001.zip*
-rwxrwxrwx 1 1018 1018 2.0G Jul 17 13:15 WRF-20250716T095739Z-1-002.zip*
-rwxrwxrwx 1 1018 1018 2.0G Jul 17 13:14 WRF-20250716T095739Z-1-003.zip*
-rwxrwxrwx 1 1018 1018 2.0G Jul 17 13:17 WRF-20250716T095739Z-1-004.zip*
-rwxrwxrwx 1 1018 1018 2.0G Jul 17 13:16 WRF-20250716T095739Z-1-005.zip*
-rwxrwxrwx 1 root root 2.3G Jul 31 13:13 custom_concat_train.nc*
drwxrwxrwx 4 root root 4.0K Jul 17 13:30 unziped/


## Unzip the files

In [ ]:
#! apt-get update && apt-get install -y unzip

In [ ]:
#%%sh 
#for file in /app/data/custom_data_1/*.zip; do unzip "$file" -d /app/data/custom_data_1/unziped; done

## Interpolate

In [ ]:
import os
import xarray as xr
import numpy as np
from glob import glob
from scipy.interpolate import griddata
from scipy.interpolate import RectBivariateSpline

from utils.interp_era5_to_wrf import process_date

ERA5_DIR = "/app/data/custom_data_1/unziped/ERA5/"  # Folder with both pressure and single level files
WRF_DIR = "/app/data/custom_data_1/unziped/WRF/"
OUT_DIR = "/app/data/custom_data_1/ERA5_preprocessed/"
LEVELS = [500, 850]  # hPa
VARIABLES = ["t", "z", "u", "v"]  # From pressure level file
SINGLE_VARS = ["u10", "v10", "d2m", "t2m", "sst", "skt", "sp", "tcwv", "tp", "q"]       # From single level file
os.makedirs(OUT_DIR, exist_ok=True)

era5_files = sorted(glob(os.path.join(ERA5_DIR, "era5_pressure_levels_cropped_*.nc")))
date_list = [os.path.basename(f).replace("era5_pressure_levels_cropped_", "").replace(".nc", "") for f in era5_files]

for date in date_list:
    process_date(date, ERA5_DIR, WRF_DIR, OUT_DIR, VARIABLES, LEVELS, SINGLE_VARS)

## Combine 

In [ ]:
import os
import glob
import json
import numpy as np
import xarray as xr
from netCDF4 import Dataset
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from utils.merge import merge_files

ERA5_DIR = "/app/data/custom_data_1/ERA5_preprocessed/"
WRF_DIR = "/app/data/custom_data_1/unziped/WRF/"
OUT_DIR = "/app/data/custom_data_1/ERA5_wrf_combined/"

os.makedirs(OUT_DIR, exist_ok=True)

merge_files(ERA5_DIR, WRF_DIR, OUT_DIR)

## Compute stats

In [ ]:
from utils.merge import compute_dataset_stats

JSON_OUT = os.path.join(OUT_DIR, "uae_stats.json")
INVARIANT_FILE = None #os.path.join(WRF_DIR, "invariants_wrf.nc")
invariant_ds = None #xr.open_dataset(INVARIANT_FILE)[["XLAT", "XLONG", "elev_mean", "lsm_mean", "land_use"]]

compute_dataset_stats(OUT_DIR, JSON_OUT, invariant_ds=invariant_ds)

## Concat

In [ ]:
from utils.concat import concat
import os

# Parameters
OUT_DIR = "/app/data/custom_data_1/ERA5_wrf_combined/"
OUTPUT_FILE = os.path.join(OUT_DIR, "custom_concat_train.nc")
MULTIPLE_OF = 8  # Can be set to 2, 4, 8, etc.
PAD_VALUE = 0  # Value to use for padding
end = -1

concat(OUT_DIR, OUTPUT_FILE, MULTIPLE_OF, PAD_VALUE, end)

# Train regression

physicsnemo/examples/weather/corrdiff/conf/config_training_custom_regression.yaml

In [7]:
! cd /app && torchrun --nproc_per_node=8 \
    examples/weather/corrdiff/train.py \
    hydra.run.dir=/app/outputs \
    dataset.type="/app/examples/weather/corrdiff/datasets/custom.py::CustomDataset" \
    dataset.data_path="/app/data/custom_data_1/ERA5_wrf_combined/custom_concat_train.nc" \
    dataset.stats_path="/app/data/custom_data_1/ERA5_wrf_combined/uae_stats_1.json" \
    training.hp.lr=0.01 \
    --config-name=config_training_custom_regression

W0807 14:04:49.570173 1127 torch/distributed/run.py:766] 
W0807 14:04:49.570173 1127 torch/distributed/run.py:766] *****************************************
W0807 14:04:49.570173 1127 torch/distributed/run.py:766] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0807 14:04:49.570173 1127 torch/distributed/run.py:766] *****************************************
/usr/local/lib/python3.11/dist-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'config_training_custom_regression': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
/usr/local/lib/python3.11/dist-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'config_training_custom_regression': Defaults list is missing `_self_`. See https://hy

# Train Diffusion

physicsnemo/examples/weather/corrdiff/conf/config_training_custom_diffusion.yaml

In [2]:
! cd /app && torchrun --nproc_per_node=8 \
    examples/weather/corrdiff/train.py \
    hydra.run.dir=/app/outputs \
    dataset.type="/app/examples/weather/corrdiff/datasets/custom.py::CustomDataset" \
    dataset.data_path="/app/data/custom_data_1/ERA5_wrf_combined/custom_concat_train.nc" \
    dataset.stats_path="/app/data/custom_data_1/ERA5_wrf_combined/uae_stats_1.json" \
    training.io.regression_checkpoint_path="/app/checkpoints_regression/UNet.0.180000.mdlus" \
    --config-name=config_training_custom_diffusion --cfg job

W0813 06:40:10.516287 88 torch/distributed/run.py:766] 
W0813 06:40:10.516287 88 torch/distributed/run.py:766] *****************************************
W0813 06:40:10.516287 88 torch/distributed/run.py:766] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0813 06:40:10.516287 88 torch/distributed/run.py:766] *****************************************
/usr/local/lib/python3.11/dist-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'config_training_custom_diffusion': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
/usr/local/lib/python3.11/dist-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'config_training_custom_diffusion': Defaults list is missing `_self_`. See https://hydra.cc/doc

In [1]:
! cd /app && torchrun --nproc_per_node=8 \
    examples/weather/corrdiff/train.py \
    hydra.run.dir=/app/outputs \
    dataset.type="/app/examples/weather/corrdiff/datasets/custom.py::CustomDataset" \
    dataset.data_path="/app/data/custom_data_1/ERA5_wrf_combined/custom_concat_train.nc" \
    dataset.stats_path="/app/data/custom_data_1/ERA5_wrf_combined/uae_stats_1.json" \
    training.io.regression_checkpoint_path="/app/checkpoints_regression/UNet.0.180000.mdlus" \
    --config-name=config_training_custom_diffusion

W0813 06:48:21.978704 121 torch/distributed/run.py:766] 
W0813 06:48:21.978704 121 torch/distributed/run.py:766] *****************************************
W0813 06:48:21.978704 121 torch/distributed/run.py:766] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0813 06:48:21.978704 121 torch/distributed/run.py:766] *****************************************
/usr/local/lib/python3.11/dist-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'config_training_custom_diffusion': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
/usr/local/lib/python3.11/dist-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'config_training_custom_diffusion': Defaults list is missing `_self_`. See https://hydra.cc